# Autodiff Playground

An interactive tour of the public API in `autodiff`. Run the cells from top to bottom, then change the values and expressions to experiment. The notebook uses only NumPy and this repository's package.

Open this notebook from the repository root so `from autodiff import ...` works. In VS Code, choose the project Python environment (or `.venv`) as the kernel.

In [1]:
import numpy as np

from autodiff import (
    Tensor, add, apply_mask, check_gradient, div, gelu, matmul, mean,
    mul, neg, power, relu, sigmoid, softmax, softmax_cross_entropy,
    sub, sum, tanh,
)

np.set_printoptions(precision=4, suppress=True)
print("Autodiff playground is ready")

Autodiff playground is ready


## 1. Tensors and their metadata

`Tensor` stores a float64 NumPy array. Set `requires_grad=True` on values whose gradients you want to inspect.

In [2]:
x = Tensor([[1, 2, 3], [4, 5, 6]], requires_grad=True)
print(x)
print("data:\n", x.data)
print("shape:", x.shape, "ndim:", x.ndim)
print("requires_grad:", x.requires_grad, "grad before backward:", x.grad)

Tensor(data=array([[1., 2., 3.],
       [4., 5., 6.]]), requires_grad=True, grad=None)
data:
 [[1. 2. 3.]
 [4. 5. 6.]]
shape: (2, 3) ndim: 2
requires_grad: True grad before backward: None


## 2. Arithmetic operators and function equivalents

The operators and the named functions below both construct differentiable tensors. NumPy arrays and Python scalars can be mixed with tensors; broadcasting is supported.

In [3]:
a = Tensor([2.0, 4.0], requires_grad=True)
b = Tensor([3.0, 5.0], requires_grad=True)

examples = {
    "a + b": a + b,
    "add(a, b)": add(a, b),
    "a - b": a - b,
    "sub(a, b)": sub(a, b),
    "a * b": a * b,
    "mul(a, b)": mul(a, b),
    "a / b": a / b,
    "div(a, b)": div(a, b),
    "-a": -a,
    "neg(a)": neg(a),
    "a ** 2": a ** 2,
    "power(a, 2)": power(a, 2),
    "10 - a (reverse subtraction)": 10 - a,
    "2 / a (reverse division)": 2 / a,
    "3 + a (reverse addition)": 3 + a,
    "2 * a (reverse multiplication)": 2 * a,
}
for name, result in examples.items():
    print(f"{name:32} -> {result.data}")

a + b                            -> [5. 9.]
add(a, b)                        -> [5. 9.]
a - b                            -> [-1. -1.]
sub(a, b)                        -> [-1. -1.]
a * b                            -> [ 6. 20.]
mul(a, b)                        -> [ 6. 20.]
a / b                            -> [0.6667 0.8   ]
div(a, b)                        -> [0.6667 0.8   ]
-a                               -> [-2. -4.]
neg(a)                           -> [-2. -4.]
a ** 2                           -> [ 4. 16.]
power(a, 2)                      -> [ 4. 16.]
10 - a (reverse subtraction)     -> [8. 6.]
2 / a (reverse division)         -> [1.  0.5]
3 + a (reverse addition)         -> [5. 7.]
2 * a (reverse multiplication)   -> [4. 8.]


In [7]:
# Broadcasting a vector across rows; backward sums the broadcast contributions.
matrix = Tensor([[1.0, 2.0], [3.0, 4.0]], requires_grad=True)
bias = Tensor([0.5, -1.0], requires_grad=True)
loss = sum(matrix + 2*bias)
loss.backward()
print("broadcast result:\n", (matrix + bias).data)
print("matrix gradient:\n", matrix.grad)
print("bias gradient:", bias.grad)

broadcast result:
 [[1.5 1. ]
 [3.5 3. ]]
matrix gradient:
 [[1. 1.]
 [1. 1.]]
bias gradient: [4. 4.]


## 3. Matrix multiplication

`@` and `matmul` support vector/matrix multiplication and batched NumPy-compatible matrix multiplication.

In [8]:
features = Tensor([[1.0, 2.0], [3.0, 1.0]], requires_grad=True)
weights = Tensor([[0.2, -0.5, 1.0], [0.4, 0.3, -0.2]], requires_grad=True)
scores = features @ weights
print("operator output:\n", scores.data)
print("function output:\n", matmul(features, weights).data)
print("reverse matmul output:", (np.array([1.0, 0.5]) @ weights).data)
mean(scores).backward()
print("features gradient:\n", features.grad)
print("weights gradient:\n", weights.grad)

operator output:
 [[ 1.   0.1  0.6]
 [ 1.  -1.2  2.8]]
function output:
 [[ 1.   0.1  0.6]
 [ 1.  -1.2  2.8]]
reverse matmul output: [ 0.4  -0.35  0.9 ]
features gradient:
 [[0.1167 0.0833]
 [0.1167 0.0833]]
weights gradient:
 [[0.6667 0.6667 0.6667]
 [0.5    0.5    0.5   ]]


## 4. Backpropagation, explicit seeds, accumulation, and clearing gradients

An implicit seed of 1 is available only for scalar outputs. For a non-scalar output, provide a gradient with the same shape. Calling `backward` again accumulates into existing `.grad` values; `zero_grad()` clears a tensor and every ancestor in its graph.

In [49]:
x = Tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x * x

y.parents
y.op

import inspect

# Assuming 'fn' is your lambda function reference
print(inspect.getsource(y._backward))

        backward = lambda grad: (
            _unbroadcast(grad * b_data, a.shape) if a.requires_grad else None,
            _unbroadcast(grad * a_data, b.shape) if b.requires_grad else None,
        )



In [50]:
y.backward(np.array([1.0, 10.0]))

ValueError: Gradient shape (2,) does not match tensor shape (3,)

In [51]:
y.backward(np.array([1.0, 10.0, 100.0]))

Final Result After the Loop Completes
When the stack is completely empty, the variables hold these exact values:

topo (List of Tensors):

Contains the tensors in reverse-topological order (dependencies first, ending with the root):

[a, b, z, y]

visited (Set of Tensors):

Contains every unique tensor in the computational graph to ensure no node is processed twice:

{a, b, z, y}

stack (List of Tuples):

Empty ([]), because all nodes have been popped, processed, and popped a second time.

Why This Design Is Used (Instead of Recursion)
Avoids Recursion Limits: Standard DFS is often written using recursive functions. If a neural network has hundreds of layers deep, recursion will crash Python with a RecursionError: maximum recursion depth exceeded.

Explicit Control: Using an explicit stack list mimics the call stack manually, allowing you to traverse massive computational graphs safely and efficiently in a flat while loop.

In [72]:
x = Tensor([1.0, 2.0, 3.0], requires_grad=True)
z = Tensor([2.0, 2.0, 3.0], requires_grad=True)
a = Tensor([1.0, 3.0, 3.0], requires_grad=True)
y = x * z * a
y.backward(np.array([1.0, 10.0, 100.0]))
print("y = x squared:", y.data)
print("seeded gradient (2*x times seed):", x.grad)
print("y.grad:", y.grad)
print("z.grad:", z.grad)
print("a.grad:", a.grad)

y = x squared: [ 2. 12. 27.]
seeded gradient (2*x times seed): [  2.  60. 900.]
y.grad: [  1.  10. 100.]
z.grad: [  1.  60. 900.]
a.grad: [  2.  40. 900.]


In [67]:
# Shared subexpressions accumulate both paths: d(x*x + 3*x)/dx = 2*x + 3.
x = Tensor(2.0, requires_grad=True)
shared_loss = x * x + 3 * x
shared_loss.backward()
print("loss:", shared_loss.data.item(), "gradient:", x.grad.item())

# Re-running backward accumulates another copy of the same gradient.
shared_loss.backward()
print("after a second backward:", x.grad.item())

shared_loss.backward()
print("after a third backward:", x.grad.item())

shared_loss.zero_grad()
print("after zero_grad on output:", x.grad, shared_loss.grad)

loss: 10.0 gradient: 7.0
after a second backward: 14.0
after a third backward: 21.0
after zero_grad on output: None None


In [12]:
# This is the expected error: implicit gradients require a scalar output.
try:
    Tensor([1.0, 2.0], requires_grad=True).backward()
except RuntimeError as error:
    print(type(error).__name__ + ":", error)

RuntimeError: An implicit gradient can only be created for a scalar output


## 5. Sum and mean reductions

Use `axis` to reduce selected dimensions and `keepdims=True` to preserve those dimensions. `axis=None` reduces all elements; multiple axes can be supplied as a tuple.

In [13]:
values = Tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], requires_grad=True)
print("sum all:", sum(values).data)
print("mean all:", mean(values).data)
print("sum over rows (axis=0):", sum(values, axis=0).data)
print("mean over columns (axis=1, keepdims):\n", mean(values, axis=1, keepdims=True).data)
print("sum axes (0, 1):", sum(values, axis=(0, 1)).data)

mean(values, axis=0).backward(np.array([1.0, 2.0, 3.0]))
print("gradient of this vector-Jacobian product:\n", values.grad)

sum all: 21.0
mean all: 3.5
sum over rows (axis=0): [5. 7. 9.]
mean over columns (axis=1, keepdims):
 [[2.]
 [5.]]
sum axes (0, 1): 21.0
gradient of this vector-Jacobian product:
 [[0.5 1.  1.5]
 [0.5 1.  1.5]]


## 6. Activation functions

Each activation is available as a public function and records a backward rule.

In [14]:
activation_input = Tensor([-2.0, -0.5, 0.0, 0.5, 2.0], requires_grad=True)
for name, activation in [("relu", relu), ("tanh", tanh), ("sigmoid", sigmoid), ("gelu", gelu)]:
    result = activation(activation_input)
    print(f"{name:8} -> {result.data}")

sum(gelu(activation_input)).backward()
print("GELU gradient:", activation_input.grad)

relu     -> [0.  0.  0.  0.5 2. ]
tanh     -> [-0.964  -0.4621  0.      0.4621  0.964 ]
sigmoid  -> [0.1192 0.3775 0.5    0.6225 0.8808]
gelu     -> [-0.0454 -0.1543  0.      0.3457  1.9546]
GELU gradient: [-0.0861  0.1326  0.5     0.8674  1.0861]


## 7. Softmax and softmax cross-entropy

`softmax` normalizes the last axis. `softmax_cross_entropy` expects logits and a same-shaped target distribution (often one-hot); it returns the mean loss across rows and uses a numerically stable fused computation.

In [15]:
logits = Tensor([[2.0, 1.0, 0.1], [0.2, 0.4, 1.4]], requires_grad=True)
probabilities = softmax(logits)
print("probabilities:\n", probabilities.data)
print("row sums:", probabilities.data.sum(axis=-1))

targets = np.array([[1.0, 0.0, 0.0], [0.0, 0.0, 1.0]])
loss = softmax_cross_entropy(logits, targets)
loss.backward()
print("mean cross-entropy:", loss.data.item())
print("logit gradient:\n", logits.grad)

probabilities:
 [[0.659  0.2424 0.0986]
 [0.1805 0.2204 0.5991]]
row sums: [1. 1.]
mean cross-entropy: 0.46464939502596025
logit gradient:
 [[-0.1705  0.1212  0.0493]
 [ 0.0902  0.1102 -0.2004]]


Try soft targets too: every target row may be a probability distribution, so it need not be strictly one-hot.

In [16]:
soft_targets = np.array([[0.8, 0.2, 0.0], [0.1, 0.2, 0.7]])
print("soft-target loss:", softmax_cross_entropy(logits, soft_targets).data.item())

soft-target loss: 0.7246493950259603


## 8. Fixed binary masks

`apply_mask(weights, mask)` applies a broadcastable zero/one mask. Gradients at masked positions are zero.

In [19]:
weights = Tensor([[0.2, -0.3, 0.5], [0.7, 0.1, -0.4]], requires_grad=True)

In [32]:
weights.parents

()

In [28]:
weights = Tensor([[0.2, -0.3, 0.5], [0.7, 0.1, -0.4]], requires_grad=True)
mask = np.array([[1, 0, 1]])  # broadcast across both rows
effective_weights = apply_mask(weights, mask)
print("effective weights:\n", effective_weights.data)
sum(effective_weights).backward()
print("weight gradient (masked column is zero):\n", weights.grad)

effective weights:
 [[ 0.2 -0.   0.5]
 [ 0.7  0.  -0.4]]
weight gradient (masked column is zero):
 [[1. 0. 1.]
 [1. 0. 1.]]


## 9. Checking derivatives numerically

`check_gradient(fn, values, eps=..., tol=...)` compares autodiff to central finite differences for a scalar-output function. It returns the relative error or raises `AssertionError` if the error is not below the tolerance.

In [18]:
def tiny_objective(x):
    return mean((x * x + 2 * x) ** 2)

relative_error = check_gradient(tiny_objective, np.array([-0.7, 0.2, 1.1]))
print("gradient check relative error:", relative_error)

gradient check relative error: 2.1172507844888154e-11


## 10. Small forward/backward example

This miniature classifier combines matrix multiplication, bias broadcasting, ReLU, softmax cross-entropy, and parameter gradients. Change the weights or inputs and rerun.

In [ ]:
inputs = Tensor([[1.0, 0.5], [-1.0, 2.0]], requires_grad=False)
w1 = Tensor([[0.2, -0.4, 0.1], [0.7, 0.3, -0.2]], requires_grad=True)
b1 = Tensor([0.1, 0.0, 0.2], requires_grad=True)
targets = np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]])

hidden = relu(inputs @ w1 + b1)
loss = softmax_cross_entropy(hidden, targets)
loss.backward()
print("hidden activations:\n", hidden.data)
print("loss:", loss.data.item())
print("w1 gradient:\n", w1.grad)
print("b1 gradient:", b1.grad)

## 11. Your turn

1. Change `a` and `b` above to arrays of different shapes that broadcast, then predict the gradient shapes before running backward.
2. Build a scalar function using at least three different operations and compare its gradient with `check_gradient`.
3. Try reducing along a different axis and use an explicit seed to select a vector-Jacobian product.
4. Add a second matrix multiplication and activation to the classifier. Which parameter gradients change?
5. Change the mask and confirm which weight gradients become zero.